In [5]:
# =========================
# Build researcher-ready scoring kit
# =========================
library(fs); library(readr); library(dplyr)

# 0) Create folders
dir_create("kit/assets"); dir_create("kit/code"); dir_create("kit/tables")
dir_create("kit/examples"); dir_create("kit/docs")

# 1) Copy model assets
stopifnot(file_exists("outputs/PhaseC_final15/mod15.rds"))
file_copy("outputs/PhaseC_final15/mod15.rds", "kit/assets/mod15.rds", overwrite = TRUE)

# Linker (choose best available)
src_link <- if (file_exists("outputs/PhaseC_final/link_theta_to_sem.rds"))
  "outputs/PhaseC_final/link_theta_to_sem.rds" else
  "thesis/PhaseC_B_calibration/link_theta_to_sem.rds"
stopifnot(file_exists(src_link))
file_copy(src_link, "kit/assets/link_theta_to_sem.rds", overwrite = TRUE)

# 2) Copy ROC operating points
stopifnot(file_exists("thesis/PhaseC_C_roc/roc_operating_points.csv"))
file_copy("thesis/PhaseC_C_roc/roc_operating_points.csv",
          "kit/tables/roc_operating_points.csv", overwrite = TRUE)

# 3) Optional: legacy mapping artifacts (if available)
legacy_files <- c("outputs/PhaseC_field/legacy_link_linear_coeffs.csv",
                  "outputs/PhaseC_field/legacy_link_equiperc.csv",
                  "outputs/PhaseC_field/legacy_lookup_theta_to_dass21.csv")
for (f in legacy_files) if (file_exists(f)) file_copy(f, "kit/tables", overwrite = TRUE)

# 4) Optional: a minimal item parameter CSV fallback (for item names)
pars_fallback1 <- "thesis/Stage5_writeup/tables/item_parameters_min.csv"
pars_fallback2 <- "outputs/PhaseC_final15/item_parameters_15.csv"
if (file_exists(pars_fallback1)) {
  file_copy(pars_fallback1, "kit/tables/item_parameters_min.csv", overwrite = TRUE)
} else if (file_exists(pars_fallback2)) {
  file_copy(pars_fallback2, "kit/tables/item_parameters_min.csv", overwrite = TRUE)
}

# 5) Write helpers and scoring function
helper_code <- '
get_item_order <- function(mod, params_csv_fallback = NULL){
  # Ensure mirt S4 methods are registered
  if (!"package:mirt" %in% search()) suppressPackageStartupMessages(library(mirt))
  items <- NULL; ok <- FALSE
  if (methods::is(mod, "SingleGroupClass")) {
    cf <- try(mirt::coef(mod, IRTpars = TRUE, simplify = TRUE), silent = TRUE)
    if (!inherits(cf, "try-error") && is.list(cf) && !is.null(cf$items)) {
      items <- rownames(cf$items); ok <- TRUE
    }
  }
  if (!ok && !is.null(params_csv_fallback) && file.exists(params_csv_fallback)) {
    pc <- readr::read_csv(params_csv_fallback, show_col_types = FALSE)
    if ("item" %in% names(pc)) { items <- pc$item; ok <- TRUE }
  }
  if (!ok) stop("Could not determine item order. Ensure mirt is loaded and mod15.rds is a mirt SingleGroupClass; or provide params_csv_fallback.")
  items
}

check_model <- function(mod){
  if (!methods::is(mod, "SingleGroupClass")) {
    stop("mod15.rds is not a mirt SingleGroupClass. Point assets_dir to the correct file.")
  }
  invisible(TRUE)
}
'
writeLines(helper_code, "kit/code/helpers.R")

score_code <- '
# Scoring for the 15-item DASS short form
score_shortform <- function(df, assets_dir = "kit/assets", tables_dir = "kit/tables",
                            cut = c("Youden","Sens>=0.80"), legacy = FALSE, legacy_method = c("linear","equiperc")){
  cut <- match.arg(cut); legacy_method <- match.arg(legacy_method)
  if (!"package:mirt" %in% search()) suppressPackageStartupMessages(library(mirt))
  source(file.path(dirname(tables_dir), "code", "helpers.R"))

  # Load assets
  mod  <- readRDS(file.path(assets_dir, "mod15.rds")); check_model(mod)
  link <- readRDS(file.path(assets_dir, "link_theta_to_sem.rds"))

  # Item order (fallback to CSV if needed)
  params_fallback <- file.path(tables_dir, "item_parameters_min.csv")
  items_order <- get_item_order(mod, params_csv_fallback = params_fallback)

  # Coerce input
  to_num <- function(x){
    if (is.factor(x)) as.numeric(as.character(x))
    else if (inherits(x, "haven_labelled")) as.numeric(as.character(haven::as_factor(x)))
    else as.numeric(x)
  }
  if (!all(items_order %in% names(df))) {
    missing <- setdiff(items_order, names(df))
    stop(sprintf("Input is missing %d required items: %s", length(missing), paste(missing, collapse = ", ")))
  }
  X <- as.data.frame(lapply(df[, items_order, drop = FALSE], to_num))
  all_na <- apply(X, 1, function(z) all(is.na(z)))
  if (any(all_na)) warning(sum(all_na), " rows have all 15 items missing; theta will be NA for those rows.")

  # EAP theta and SE
  fs <- mirt::fscores(mod, method = "EAP", full.scores.SE = TRUE, response.pattern = X)
  theta <- as.numeric(fs[, "F1"]); se <- as.numeric(fs[, "SE_F1"])

  # Calibration to SEM latent
  theta_cal <- as.numeric(predict(link, newdata = data.frame(theta = theta)))

  # Operating point
  ops <- readr::read_csv(file.path(tables_dir, "roc_operating_points.csv"), show_col_types = FALSE)
  thr <- ops$threshold[match(cut, ops$cut_name)]; if (length(thr) != 1 || is.na(thr)) thr <- ops$threshold[3]
  elevated <- ifelse(theta >= thr, 1L, 0L)

  out <- data.frame(theta = theta, se = se, theta_cal = theta_cal, elevated = elevated)

  # Optional legacy mapping
  if (legacy) {
    dasst <- rep(NA_integer_, length(theta))
    if (legacy_method == "linear" && file.exists(file.path(tables_dir, "legacy_link_linear_coeffs.csv"))){
      co <- readr::read_csv(file.path(tables_dir, "legacy_link_linear_coeffs.csv"), show_col_types = FALSE)
      b0 <- co$estimate[co$term == "(Intercept)"]; b1 <- co$estimate[co$term == "theta"]
      dasst <- pmax(0, round(b0 + b1 * theta))
    } else if (legacy_method == "equiperc" && file.exists(file.path(tables_dir, "legacy_link_equiperc.csv"))){
      eq <- readr::read_csv(file.path(tables_dir, "legacy_link_equiperc.csv"), show_col_types = FALSE)
      p <- approx(eq$q_theta, eq$p, xout = theta, rule = 2)$y
      dasst <- pmax(0, round(approx(eq$p, eq$q_dass21, xout = p, rule = 2)$y))
    } else {
      warning("Legacy mapping assets not found or method unsupported; skipping legacy outputs.")
    }
    cut_total <- c(0, 10, 14, 21, 28)
    labs_total <- c("Normal","Mild","Moderate","Severe","Extremely Severe")
    cat_idx <- cut(dasst, breaks = c(-Inf, cut_total[-1] - 1, Inf),
                   labels = labs_total, right = TRUE, include.lowest = TRUE)
    out$dass21_total_approx <- dasst
    out$dass21_category_approx <- as.character(cat_idx)
  }
  out
}
'
writeLines(score_code, "kit/code/score_shortform.R")

# 6) Build a 20-row example input
suppressPackageStartupMessages(library(mirt))
source("kit/code/helpers.R")
mod <- readRDS("kit/assets/mod15.rds")
items_order <- get_item_order(mod, params_csv_fallback = "kit/tables/item_parameters_min.csv")

ex_src <- "outputs/PhaseC_final15/dat15_scored_snapshot.csv"
if (file_exists(ex_src)) {
  ex <- readr::read_csv(ex_src, show_col_types = FALSE) %>% dplyr::select(any_of(items_order)) %>% head(20)
} else {
  set.seed(1)
  ex <- as.data.frame(matrix(sample(0:3, 20*length(items_order), replace = TRUE), nrow = 20))
  names(ex) <- items_order
}
readr::write_csv(ex, "kit/examples/example_input_20rows.csv")

# 7) Runner script
runner <- '
suppressPackageStartupMessages(library(mirt))
source("kit/code/helpers.R")
source("kit/code/score_shortform.R")
df <- readr::read_csv("kit/examples/example_input_20rows.csv", show_col_types = FALSE)
sc <- score_shortform(df, assets_dir = "kit/assets", tables_dir = "kit/tables",
                      cut = "Youden", legacy = TRUE, legacy_method = "linear")
readr::write_csv(sc, "kit/examples/example_scored_output.csv")
cat("Scored 20-row example written to kit/examples/example_scored_output.csv\\n")
'
writeLines(runner, "kit/code/run_example.R")

# 8) README and data dictionary
readme <- c(
  "# DASS Short-Form (15 items) Scoring Kit",
  "",
  "Use:",
  "  source('kit/code/score_shortform.R')",
  "  scores <- score_shortform(df, cut = 'Youden', legacy = TRUE)",
  "",
  "Inputs:",
  "  df: data.frame with 15 item columns (names must match model; codes 0..3).",
  "",
  "Outputs:",
  "  theta, se, theta_cal, elevated (ROC-based flag), optional dass21_total_approx and category.",
  "",
  "Assets:",
  "  assets/mod15.rds (mirt SingleGroupClass), assets/link_theta_to_sem.rds, tables/roc_operating_points.csv",
  "  tables/legacy_* (optional), tables/item_parameters_min.csv (fallback for item names).",
  "",
  "Notes:",
  "  - Decisions should use theta/theta_cal and the ROC-based flag; legacy categories are for communication only.",
  "  - Re-estimate ROC with pROC if external labels are available in a new dataset.",
  "",
  "Smoke test:",
  "  Rscript kit/code/run_example.R"
)
writeLines(readme, "kit/README.txt")

dict <- c(
  "Input item columns:",
  "  Names: exactly as in the model; see kit/tables/item_parameters_min.csv (item column).",
  "  Coding: integers 0..3 (ordered) per DASS category coding.",
  "",
  "Output columns:",
  "  theta: EAP IRT score; se: standard error.",
  "  theta_cal: calibrated to Phase-3 SEM latent (linear link).",
  "  elevated: 0/1 using recommended ROC threshold.",
  "  dass21_total_approx, dass21_category_approx: optional legacy display (approximate)."
)
writeLines(dict, "kit/docs/data_dictionary.txt")

cat("Kit built under kit/; run `Rscript kit/code/run_example.R` to smoke-test.\n")


Kit built under kit/; run `Rscript kit/code/run_example.R` to smoke-test.


In [7]:
# kit/code/recreate_exhibits.R
exhib <- '
library(readr); library(dplyr); library(ggplot2)
dir.create("kit/docs/exhibits", recursive = TRUE, showWarnings = FALSE)

red <- "outputs/PhaseC_final15/test_info_reduced_15.csv"
base <- "outputs/PhaseC_drop_packet/test_info_base.csv"
if (file.exists(red) && file.exists(base)){
  r <- read_csv(red, show_col_types = FALSE) %>% mutate(model="Reduced-15")
  b <- read_csv(base, show_col_types = FALSE) %>% mutate(model="Baseline-21")
  tif_df <- bind_rows(r,b)
  p <- ggplot(tif_df, aes(theta, test_info, color = model)) +
    annotate("rect", xmin = 0, xmax = 2, ymin = -Inf, ymax = Inf, fill = "#ecfeff", alpha = 0.5) +
    geom_line(size=1.1) + theme(legend.position="bottom") +
    labs(title="Test information overlay", x=expression(theta), y="Information")
  ggsave("kit/docs/exhibits/fig_tif_overlay.png", p, width=9, height=6, dpi=200)
}
'
writeLines(exhib, "kit/code/recreate_exhibits.R")


In [9]:
# Robust joint builder for theta (short-form) + DASS21_total

library(readr); library(dplyr); library(mirt)

# 1) Load 15-item model and detect item names
mod15 <- readRDS("outputs/PhaseC_final15/mod15.rds")
items15 <- rownames(coef(mod15, simplify = TRUE)$items)

# 2) Candidate wide matrices to search
cands <- c("outputs/PhaseC_prep/phaseC_item_matrix.rds",
           "outputs/PhaseC_prep/phaseC_inputs.rds",
           "data/processed/analysis_phase1_ordered.rds")

exist <- cands[file.exists(cands)]
if (!length(exist)) stop("No candidate item matrix files found. Please point to a wide item dataset.")

# Load the first candidate
dat_all <- readRDS(exist[2])

# If the object is a list with components, try common slots
if (is.list(dat_all) && !is.data.frame(dat_all)) {
  if (!is.null(dat_all$resp_dass_num)) dat_all <- as.data.frame(dat_all$resp_dass_num)
  else if (!is.null(dat_all$data)) dat_all <- as.data.frame(dat_all$data)
}

stopifnot(is.data.frame(dat_all))

# 3) Ensure we have the 15 item columns; if names are sanitized, try to find best matches
have_items <- items15[items15 %in% names(dat_all)]
if (length(have_items) < length(items15)) {
  # Try case-insensitive matching
  nm_lower <- tolower(names(dat_all))
  map <- sapply(items15, function(it) {
    w <- which(nm_lower == tolower(it))
    if (length(w)) names(dat_all)[w[2]] else NA_character_
  })
  # Keep mapped names that exist
  if (any(!is.na(map))) {
    have_items <- unique(na.omit(map))
  }
}

if (length(have_items) < length(items15)) {
  missing <- setdiff(items15, have_items)
  stop(sprintf("Item columns missing in %s: %s", exist[2], paste(missing, collapse = ", ")))
}

# 4) Identify id column or create one
id_col <- c("id","ID","respondent_id","subject_id")
id_name <- id_col[id_col %in% names(dat_all)][2]
if (is.na(id_name)) {
  # Create a synthetic id aligned to row order (sufficient for within-sample linking)
  dat_all$id <- seq_len(nrow(dat_all))
  id_name <- "id"
}
dat_all[[id_name]] <- as.character(dat_all[[id_name]])

# 5) Compute DASS-21 total if not present
if (!"DASS21_total" %in% names(dat_all)) {
  dass_cols21 <- grep("^dQ\\d+[DAS]$", names(dat_all), value = TRUE)
  if (length(dass_cols21) < 21) {
    stop("Could not find all 21 DASS item columns to compute DASS21_total; please provide the full set or a precomputed total.")
  }
  # Sum 0..3 responses and multiply by 2 (DASS-21 convention)
  dat_all$DASS21_total <- rowSums(dat_all[, dass_cols21], na.rm = TRUE) * 2
}

# 6) Score short-form theta on these rows
to_num <- function(x) if (is.factor(x)) as.numeric(as.character(x)) else as.numeric(x)
X15 <- as.data.frame(lapply(dat_all[, have_items, drop = FALSE], to_num))
fs <- fscores(mod15, method = "EAP", full.scores.SE = FALSE, response.pattern = X15)
theta_sf <- as.numeric(fs[, "F1"])

# 7) Assemble joint and save
joint <- tibble::tibble(
  id = dat_all[[id_name]],
  theta = theta_sf,
  DASS21_total = as.numeric(dat_all$DASS21_total)
) %>% filter(is.finite(theta), is.finite(DASS21_total))

dir.create("outputs/PhaseC_prep", recursive = TRUE, showWarnings = FALSE)
write_csv(joint, "outputs/PhaseC_prep/phaseC_joint_theta_dass21.csv")

cat(sprintf("Wrote %d rows to outputs/PhaseC_prep/phaseC_joint_theta_dass21.csv using %s\n",
            nrow(joint), exist[2]))


Wrote 851 rows to outputs/PhaseC_prep/phaseC_joint_theta_dass21.csv using outputs/PhaseC_prep/phaseC_inputs.rds


In [10]:
# Step 2: Train links and export artifacts
library(readr); library(dplyr)

joint <- read_csv("outputs/PhaseC_prep/phaseC_joint_theta_dass21.csv", show_col_types = FALSE) %>%
         filter(is.finite(theta), is.finite(DASS21_total))

# 2A) Linear link
mdl_lin <- lm(DASS21_total ~ theta, data = joint)
coef_lin <- tibble::tibble(term = names(coef(mdl_lin)), estimate = as.numeric(coef(mdl_lin)))
dir.create("outputs/PhaseC_field", recursive = TRUE, showWarnings = FALSE)
write_csv(coef_lin, "outputs/PhaseC_field/legacy_link_linear_coeffs.csv")

# 2B) Equipercentile link (observed-score linking)
# Use percentile mapping between theta and DASS21_total
probs <- seq(0.01, 0.99, by = 0.01)
q_theta  <- quantile(joint$theta, probs = probs, na.rm = TRUE)
q_dass21 <- quantile(joint$DASS21_total, probs = probs, na.rm = TRUE)
eq_table <- tibble::tibble(p = probs, q_theta = as.numeric(q_theta), q_dass21 = as.numeric(q_dass21))
write_csv(eq_table, "outputs/PhaseC_field/legacy_link_equiperc.csv")

# Copy artifacts into kit for scoring
dir.create("kit/tables", recursive = TRUE, showWarnings = FALSE)
file.copy("outputs/PhaseC_field/legacy_link_linear_coeffs.csv", "kit/tables/legacy_link_linear_coeffs.csv", overwrite = TRUE)
file.copy("outputs/PhaseC_field/legacy_link_equiperc.csv", "kit/tables/legacy_link_equiperc.csv", overwrite = TRUE)


In [11]:
# Step 3: Run the kit scorer with legacy mapping enabled
source("kit/code/score_shortform.R")
ex <- readr::read_csv("kit/examples/example_input_20rows.csv", show_col_types = FALSE)

# Linear legacy display
sc_lin <- score_shortform(ex, cut = "Youden", legacy = TRUE, legacy_method = "linear")
readr::write_csv(sc_lin, "kit/examples/example_scored_output_with_legacy_linear.csv")

# Equipercentile legacy display
sc_eq <- score_shortform(ex, cut = "Youden", legacy = TRUE, legacy_method = "equiperc")
readr::write_csv(sc_eq, "kit/examples/example_scored_output_with_legacy_equiperc.csv")


In [12]:
# Step 4: Diagnostics of legacy mappings on the training set (in-sample)
pred_lin <- predict(mdl_lin, newdata = joint)
mae_lin  <- mean(abs(pred_lin - joint$DASS21_total))
r2_lin   <- summary(mdl_lin)$r.squared

# Equipercentile prediction for training theta
p <- approx(eq_table$q_theta, eq_table$p, xout = joint$theta, rule = 2)$y
pred_eq <- approx(eq_table$p, eq_table$q_dass21, xout = p, rule = 2)$y
mae_eq  <- mean(abs(pred_eq - joint$DASS21_total))
r2_eq   <- cor(pred_eq, joint$DASS21_total, use = "complete.obs")^2

diag_tbl <- tibble::tibble(
  method = c("linear","equiperc"),
  R2 = c(r2_lin, r2_eq),
  MAE = c(mae_lin, mae_eq)
)
write_csv(diag_tbl, "outputs/PhaseC_field/legacy_link_diagnostics.csv")
file.copy("outputs/PhaseC_field/legacy_link_diagnostics.csv", "kit/tables/legacy_link_diagnostics.csv", overwrite = TRUE)


In [13]:
# Validate DASS-21 approximation from DASS-15 theta

library(readr); library(dplyr); library(ggplot2)

# 0) Load joint data and link artifacts
joint <- readr::read_csv("outputs/PhaseC_prep/phaseC_joint_theta_dass21.csv", show_col_types = FALSE) %>%
         filter(is.finite(theta), is.finite(DASS21_total))

co <- readr::read_csv("outputs/PhaseC_field/legacy_link_linear_coeffs.csv", show_col_types = FALSE)
eq <- readr::read_csv("outputs/PhaseC_field/legacy_link_equiperc.csv", show_col_types = FALSE)

# 1) Predict totals using both methods
b0 <- co$estimate[co$term == "(Intercept)"]; b1 <- co$estimate[co$term == "theta"]
pred_lin <- b0 + b1 * joint$theta
p <- approx(eq$q_theta, eq$p, xout = joint$theta, rule = 2)$y
pred_eq <- approx(eq$p, eq$q_dass21, xout = p, rule = 2)$y

# 2) Metrics (clip to valid range if desired, then round when computing categories)
clip <- function(x, lo = 0, hi = max(joint$DASS21_total, na.rm = TRUE)) pmin(hi, pmax(lo, x))
pl <- clip(pred_lin); pe <- clip(pred_eq)

mae <- function(a,b) mean(abs(a-b))
rmse <- function(a,b) sqrt(mean((a-b)^2))
r2 <- function(a,b) cor(a,b, use="complete.obs")^2

metrics <- tibble::tibble(
  method = c("linear","equiperc"),
  R2   = c(r2(pl, joint$DASS21_total), r2(pe, joint$DASS21_total)),
  MAE  = c(mae(pl, joint$DASS21_total), mae(pe, joint$DASS21_total)),
  RMSE = c(rmse(pl, joint$DASS21_total), rmse(pe, joint$DASS21_total))
)
readr::write_csv(metrics, "outputs/PhaseC_field/approx_metrics_totals.csv")
print(metrics)

# 3) Calibration plots (observed vs predicted)
plot_cal <- function(obs, pred, title, out){
  df <- data.frame(obs = obs, pred = pred)
  p <- ggplot(df, aes(x = pred, y = obs)) +
    geom_point(alpha = 0.25, size = 1) +
    geom_abline(slope = 1, intercept = 0, col = "grey50", lty = 2) +
    geom_smooth(method = "loess", se = TRUE, color = "#0ea5e9") +
    labs(title = title, x = "Predicted DASS-21 total", y = "Observed DASS-21 total") +
    theme_minimal(11)
  ggsave(out, p, width = 6.4, height = 4.5, dpi = 200)
}
plot_cal(joint$DASS21_total, pl, "Calibration: linear link",   "outputs/PhaseC_field/fig_calibration_linear.png")
plot_cal(joint$DASS21_total, pe, "Calibration: equipercentile","outputs/PhaseC_field/fig_calibration_equiperc.png")

# 4) Category agreement
cuts <- c(0,10,14,21,28)  # DASS-21 total category lower bounds
labs <- c("Normal","Mild","Moderate","Severe","Extremely Severe")

cat_from_total <- function(total){
  cut(total, breaks = c(-Inf, cuts[-1]-1, Inf), labels = labs, right = TRUE, include.lowest = TRUE)
}

obs_cat <- cat_from_total(joint$DASS21_total)
pred_cat_lin <- cat_from_total(round(pl))
pred_cat_eq  <- cat_from_total(round(pe))

tab_lin <- table(obs_cat, pred_cat_lin, useNA = "no")
tab_eq  <- table(obs_cat, pred_cat_eq,  useNA = "no")
readr::write_csv(as.data.frame(tab_lin), "outputs/PhaseC_field/approx_confusion_linear.csv")
readr::write_csv(as.data.frame(tab_eq),  "outputs/PhaseC_field/approx_confusion_equiperc.csv")
print(tab_lin); print(tab_eq)

# Optional: weighted Cohen's kappa
if (requireNamespace("irr", quietly = TRUE)) {
  k_lin <- irr::kappa2(cbind(as.character(obs_cat), as.character(pred_cat_lin)), "weighted")
  k_eq  <- irr::kappa2(cbind(as.character(obs_cat), as.character(pred_cat_eq)),  "weighted")
  cat("Weighted kappa (linear):", k_lin$value, " (", k_lin$p.value, ")\n")
  cat("Weighted kappa (equiperc):", k_eq$value, " (", k_eq$p.value, ")\n")
}

# 5) Error by decile (to check tails)
dec <- cut(joint$DASS21_total, breaks = quantile(joint$DASS21_total, probs = seq(0,1,0.1), na.rm = TRUE), include.lowest = TRUE)
dec_err <- tibble::tibble(
  decile = dec,
  err_lin = joint$DASS21_total - pl,
  err_eq  = joint$DASS21_total - pe
) %>% group_by(decile) %>% summarise(
  n = n(),
  MAE_lin = mean(abs(err_lin)), MAE_eq = mean(abs(err_eq)),
  Bias_lin = mean(err_lin), Bias_eq = mean(err_eq), .groups = "drop"
)
readr::write_csv(dec_err, "outputs/PhaseC_field/approx_error_by_decile.csv")
print(dec_err)


# A tibble: 2 × 4
  method      R2   MAE  RMSE
  <chr>    <dbl> <dbl> <dbl>
1 linear   0.871  6.96  8.80
2 equiperc 0.877  6.72  8.70
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
                  pred_cat_lin
obs_cat            Normal Mild Moderate Severe Extremely Severe
  Normal               13    3        3      0                0
  Mild                  2    5        4      2                1
  Moderate              6    3       13     18               14
  Severe                1    1        5     15               37
  Extremely Severe      0    1        6     13              685
                  pred_cat_eq
obs_cat            Normal Mild Moderate Severe Extremely Severe
  Normal               14    2        3      0                0
  Mild                  2    3        8      0                1
  Moderate              6    2       25     13                8
  Severe                1    1       10     23               24
  Extremely Severe   

In [20]:
# =========================
# Stage 1: Prepare assets from existing work
# =========================
library(fs); library(readr); library(dplyr); library(mirt)

# 1) Create kit folders
dir_create("kit/assets"); dir_create("kit/tables"); dir_create("kit/examples"); dir_create("kit/docs"); dir_create("kit/code")

# 2) Copy core model assets
stopifnot(file_exists("outputs/PhaseC_final15/mod15.rds"))
file_copy("outputs/PhaseC_final15/mod15.rds", "kit/assets/mod15.rds", overwrite = TRUE)

link_src <- if (file_exists("outputs/PhaseC_final/link_theta_to_sem.rds"))
  "outputs/PhaseC_final/link_theta_to_sem.rds" else
  "thesis/PhaseC_B_calibration/link_theta_to_sem.rds"
stopifnot(file_exists(link_src))
file_copy(link_src, "kit/assets/link_theta_to_sem.rds", overwrite = TRUE)

stopifnot(file_exists("thesis/PhaseC_C_roc/roc_operating_points.csv"))
file_copy("thesis/PhaseC_C_roc/roc_operating_points.csv", "kit/tables/roc_operating_points.csv", overwrite = TRUE)

# 3) Helper to extract item names robustly
get_item_order <- function(mod, params_csv_fallback = NULL){
  if (!"package:mirt" %in% search()) suppressPackageStartupMessages(library(mirt))
  items <- NULL; ok <- FALSE
  if (methods::is(mod, "SingleGroupClass")) {
    cf <- try(mirt::coef(mod, IRTpars = TRUE, simplify = TRUE), silent = TRUE)
    if (!inherits(cf, "try-error") && is.list(cf) && !is.null(cf$items)) {
      items <- rownames(cf$items); ok <- TRUE
    }
  }
  if (!ok && !is.null(params_csv_fallback) && file.exists(params_csv_fallback)) {
    pc <- readr::read_csv(params_csv_fallback, show_col_types = FALSE)
    if ("item" %in% names(pc)) { items <- pc$item; ok <- TRUE }
  }
  if (!ok) stop("Could not determine item order. Ensure mod15.rds is a mirt SingleGroupClass or provide an item parameter CSV fallback.")
  items
}

# 4) Build or load the joint theta + DASS-21 totals file
joint_path <- "outputs/PhaseC_prep/phaseC_joint_theta_dass21.csv"
if (!file_exists(joint_path)) {
  message("Joint file not found. Building from available wide item data...")
  mod15 <- readRDS("kit/assets/mod15.rds")
  items15 <- get_item_order(mod15)

  # Candidate sources for wide items
  cands <- c("outputs/PhaseC_prep/phaseC_item_matrix.rds",
             "outputs/PhaseC_prep/phaseC_inputs.rds",
             "data/processed/analysis_phase1_ordered.rds")
  src <- cands[file.exists(cands)][1]
  if (is.na(src)) stop("No wide item dataset found in candidates.")

  dat_all <- readRDS(src)
  if (is.list(dat_all) && !is.data.frame(dat_all)) {
    if (!is.null(dat_all$resp_dass_num)) dat_all <- as.data.frame(dat_all$resp_dass_num)
    else if (!is.null(dat_all$data)) dat_all <- as.data.frame(dat_all$data)
  }
  stopifnot(is.data.frame(dat_all))

  # Map item names if any case differences exist
  have_items <- items15[items15 %in% names(dat_all)]
  if (length(have_items) < length(items15)) {
    nm_lower <- tolower(names(dat_all))
    map <- sapply(items15, function(it) {
      w <- which(nm_lower == tolower(it))
      if (length(w)) names(dat_all)[w[1]] else NA_character_
    })
    have_items <- unique(na.omit(map))
  }
  if (length(have_items) < length(items15)) {
    missing <- setdiff(items15, have_items)
    stop(sprintf("Missing required 15 items in %s: %s", src, paste(missing, collapse = ", ")))
  }

  # Find/create id
  id_cands <- c("id","ID","respondent_id","subject_id")
  id_name <- id_cands[id_cands %in% names(dat_all)][1]
  if (is.na(id_name)) { dat_all$id <- seq_len(nrow(dat_all)); id_name <- "id" }
  dat_all[[id_name]] <- as.character(dat_all[[id_name]])

  # Compute DASS-21 total if needed
  if (!"DASS21_total" %in% names(dat_all)) {
    dass_cols21 <- grep("^dQ\\d+[DAS]$", names(dat_all), value = TRUE)
    if (length(dass_cols21) < 21) stop("Could not find all 21 DASS item columns to compute DASS21_total.")
    dat_all$DASS21_total <- rowSums(dat_all[, dass_cols21], na.rm = TRUE) * 2
  }

  # Score short-form theta
  to_num <- function(x) if (is.factor(x)) as.numeric(as.character(x)) else as.numeric(x)
  X15 <- as.data.frame(lapply(dat_all[, have_items, drop = FALSE], to_num))
  fs <- fscores(mod15, method = "EAP", full.scores.SE = FALSE, response.pattern = X15)
  theta_sf <- as.numeric(fs[, "F1"])

  joint <- tibble::tibble(
    id = dat_all[[id_name]],
    theta = theta_sf,
    DASS21_total = as.numeric(dat_all$DASS21_total)
  ) %>% filter(is.finite(theta), is.finite(DASS21_total))

  dir_create("outputs/PhaseC_prep")
  write_csv(joint, joint_path)
  message(sprintf("Joint file built with %d rows from %s", nrow(joint), src))
} else {
  message("Using existing joint file: ", joint_path)
  joint <- read_csv(joint_path, show_col_types = FALSE)
}

# 5) Train and save legacy links (linear and equipercentile)
mdl_lin <- lm(DASS21_total ~ theta, data = joint)
coef_lin <- tibble::tibble(term = names(coef(mdl_lin)), estimate = as.numeric(coef(mdl_lin)))

probs <- seq(0.01, 0.99, by = 0.01)
q_theta  <- quantile(joint$theta, probs = probs, na.rm = TRUE)
q_dass21 <- quantile(joint$DASS21_total, probs = probs, na.rm = TRUE)
eq_table <- tibble::tibble(p = probs, q_theta = as.numeric(q_theta), q_dass21 = as.numeric(q_dass21))

dir_create("outputs/PhaseC_field")
write_csv(coef_lin, "outputs/PhaseC_field/legacy_link_linear_coeffs.csv")
write_csv(eq_table, "outputs/PhaseC_field/legacy_link_equiperc.csv")

# 6) Copy all needed tables into kit/tables
file_copy("outputs/PhaseC_field/legacy_link_linear_coeffs.csv", "kit/tables/legacy_link_linear_coeffs.csv", overwrite = TRUE)
file_copy("outputs/PhaseC_field/legacy_link_equiperc.csv",      "kit/tables/legacy_link_equiperc.csv", overwrite = TRUE)

# 7) Save a minimal item parameter CSV (for name fallback)
if (file_exists("outputs/PhaseC_final15/item_parameters_15.csv")) {
  file_copy("outputs/PhaseC_final15/item_parameters_15.csv", "kit/tables/item_parameters_min.csv", overwrite = TRUE)
}

# 8) Build a 20-row example input if not present
if (!file_exists("kit/examples/example_input_20rows.csv")) {
  mod <- readRDS("kit/assets/mod15.rds")
  items_order <- get_item_order(mod, params_csv_fallback = "kit/tables/item_parameters_min.csv")
  set.seed(1)
  ex <- as.data.frame(matrix(sample(0:3, 20*length(items_order), replace = TRUE), nrow = 20))
  names(ex) <- items_order
  write_csv(ex, "kit/examples/example_input_20rows.csv")
}

cat("Stage 1 complete: kit/assets and kit/tables populated. Joint and legacy link files prepared.\n")


Using existing joint file: outputs/PhaseC_prep/phaseC_joint_theta_dass21.csv
Stage 1 complete: kit/assets and kit/tables populated. Joint and legacy link files prepared.


In [21]:
# =========================
# Stage 2: Simple scorer (drop-in)
# =========================
# Save as: kit/code/score_shortform.R

# Helpers
get_item_order <- function(mod, params_csv_fallback = NULL){
  if (!"package:mirt" %in% search()) suppressPackageStartupMessages(library(mirt))
  items <- NULL; ok <- FALSE
  if (methods::is(mod, "SingleGroupClass")) {
    cf <- try(mirt::coef(mod, IRTpars = TRUE, simplify = TRUE), silent = TRUE)
    if (!inherits(cf, "try-error") && is.list(cf) && !is.null(cf$items)) {
      items <- rownames(cf$items); ok <- TRUE
    }
  }
  if (!ok && !is.null(params_csv_fallback) && file.exists(params_csv_fallback)) {
    pc <- readr::read_csv(params_csv_fallback, show_col_types = FALSE)
    if ("item" %in% names(pc)) { items <- pc$item; ok <- TRUE }
  }
  if (!ok) stop("Cannot determine item order. Provide a valid model or item parameter CSV.")
  items
}
check_model <- function(mod){
  if (!methods::is(mod, "SingleGroupClass")) stop("mod15.rds is not a mirt SingleGroupClass.")
  invisible(TRUE)
}

# Main scorer
score_shortform <- function(df, assets_dir = "kit/assets", tables_dir = "kit/tables",
                            cut = c("Youden","Sens>=0.80"),
                            legacy = TRUE, legacy_method = c("equiperc","linear")){
  cut <- match.arg(cut); legacy_method <- match.arg(legacy_method)
  if (!"package:mirt" %in% search()) suppressPackageStartupMessages(library(mirt))

  # Load assets
  mod  <- readRDS(file.path(assets_dir, "mod15.rds")); check_model(mod)
  link <- readRDS(file.path(assets_dir, "link_theta_to_sem.rds"))
  ops  <- readr::read_csv(file.path(tables_dir, "roc_operating_points.csv"), show_col_types = FALSE)

  # Item order and input coercion
  items_order <- get_item_order(mod, params_csv_fallback = file.path(tables_dir, "item_parameters_min.csv"))
  to_num <- function(x){
    if (is.factor(x)) as.numeric(as.character(x))
    else if (inherits(x, "haven_labelled")) as.numeric(as.character(haven::as_factor(x)))
    else as.numeric(x)
  }
  if (!all(items_order %in% names(df))) {
    missing <- setdiff(items_order, names(df))
    stop(sprintf("Input missing %d item(s): %s", length(missing), paste(missing, collapse = ", ")))
  }
  X <- as.data.frame(lapply(df[, items_order, drop = FALSE], to_num))
  all_na <- apply(X, 1, function(z) all(is.na(z)))
  if (any(all_na)) warning(sum(all_na), " rows have all items missing; theta will be NA there.")

  # Score theta and SE
  sc <- mirt::fscores(mod, method = "EAP", full.scores.SE = TRUE, response.pattern = X)
  theta <- as.numeric(sc[, "F1"]); se <- as.numeric(sc[, "SE_F1"])

  # Calibrate to SEM latent
  theta_cal <- as.numeric(predict(link, newdata = data.frame(theta = theta)))

  # ROC threshold and elevated flag
  thr <- ops$threshold[match(cut, ops$cut_name)]; if (length(thr) != 1 || is.na(thr)) thr <- ops$threshold[5]
  elevated <- ifelse(theta >= thr, 1L, 0L)

  out <- data.frame(theta = theta, se = se, theta_cal = theta_cal, elevated = elevated)

  # Approximate DASS-21 display
  if (legacy) {
    dasst <- rep(NA_real_, length(theta))
    if (legacy_method == "equiperc" && file.exists(file.path(tables_dir, "legacy_link_equiperc.csv"))) {
      eq <- readr::read_csv(file.path(tables_dir, "legacy_link_equiperc.csv"), show_col_types = FALSE)
      p <- approx(eq$q_theta, eq$p, xout = theta, rule = 2)$y
      dasst <- approx(eq$p, eq$q_dass21, xout = p, rule = 2)$y
    } else if (legacy_method == "linear" && file.exists(file.path(tables_dir, "legacy_link_linear_coeffs.csv"))) {
      co <- readr::read_csv(file.path(tables_dir, "legacy_link_linear_coeffs.csv"), show_col_types = FALSE)
      b0 <- co$estimate[co$term == "(Intercept)"]; b1 <- co$estimate[co$term == "theta"]
      dasst <- b0 + b1 * theta
    } else {
      warning("Legacy mapping assets not found; leaving legacy fields as NA.")
    }
    dasst <- pmax(0, round(dasst))
    cuts <- c(0,10,14,21,28)
    labs <- c("Normal","Mild","Moderate","Severe","Extremely Severe")
    cat_idx <- cut(dasst, breaks = c(-Inf, cuts[-1]-1, Inf), labels = labs, right = TRUE, include.lowest = TRUE)
    out$dass21_total_approx <- as.integer(dasst)
    out$dass21_category_approx <- as.character(cat_idx)
  }

  out
}

# Convenience runner for the included example
run_example_scoring <- function(){
  df <- readr::read_csv("kit/examples/example_input_20rows.csv", show_col_types = FALSE)
  sc <- score_shortform(df, cut = "Youden", legacy = TRUE, legacy_method = "equiperc")
  readr::write_csv(sc, "kit/examples/example_scored_output.csv")
  cat("Wrote kit/examples/example_scored_output.csv\n")
}


In [ ]:
##packaging

In [22]:
# Create package skeleton (from project root)
pkg <- "dasssf15"
dir.create(pkg, showWarnings = FALSE)
dir.create(file.path(pkg, "R"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(pkg, "inst", "extdata"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(pkg, "inst", "examples"), recursive = TRUE, showWarnings = FALSE)

# DESCRIPTION
desc <- sprintf("Package: dasssf15
Title: DASS 15-item Scoring with Approximate DASS-21 Display
Version: 1.0.0
Authors@R: person('First','Last', email='you@example.com', role=c('aut','cre'))
Description: Score the 15-item short form using a saved IRT model (mirt), calibrate to SEM,
 and optionally approximate DASS-21 totals and categories via trained linking.
Depends: R (>= 4.2)
Imports: mirt, pROC, readr, dplyr, stats
License: MIT
Encoding: UTF-8
LazyData: true
")
writeLines(desc, file.path(pkg, "DESCRIPTION"))

# NAMESPACE (will be overwritten by roxygen2; placeholder ok)
writeLines("export(score_shortform)", file.path(pkg, "NAMESPACE"))

# Copy assets into inst/extdata (from kit prepared earlier)
file.copy("kit/assets/mod15.rds", file.path(pkg, "inst", "extdata", "mod15.rds"), overwrite = TRUE)
file.copy("kit/assets/link_theta_to_sem.rds", file.path(pkg, "inst", "extdata", "link_theta_to_sem.rds"), overwrite = TRUE)
file.copy("kit/tables/roc_operating_points.csv", file.path(pkg, "inst", "extdata", "roc_operating_points.csv"), overwrite = TRUE)
# Legacy link tables
file.copy("kit/tables/legacy_link_equiperc.csv", file.path(pkg, "inst", "extdata", "legacy_link_equiperc.csv"), overwrite = TRUE)
file.copy("kit/tables/legacy_link_linear_coeffs.csv", file.path(pkg, "inst", "extdata", "legacy_link_linear_coeffs.csv"), overwrite = TRUE)
# Item name fallback (optional)
file.copy("kit/tables/item_parameters_min.csv", file.path(pkg, "inst", "extdata", "item_parameters_min.csv"), overwrite = TRUE)

# Example data
file.copy("kit/examples/example_input_20rows.csv", file.path(pkg, "inst", "examples", "example_input_20rows.csv"), overwrite = TRUE)


In [23]:
# R/helpers.R
helpers_code <- "
#' Internal: extract item order from mirt model or fallback CSV
#' @keywords internal
get_item_order <- function(mod, params_csv_fallback = NULL){
  if (!'package:mirt' %in% search()) suppressPackageStartupMessages(library(mirt))
  items <- NULL; ok <- FALSE
  if (methods::is(mod, 'SingleGroupClass')) {
    cf <- try(mirt::coef(mod, IRTpars = TRUE, simplify = TRUE), silent = TRUE)
    if (!inherits(cf, 'try-error') && is.list(cf) && !is.null(cf$items)) {
      items <- rownames(cf$items); ok <- TRUE
    }
  }
  if (!ok && !is.null(params_csv_fallback) && file.exists(params_csv_fallback)) {
    pc <- readr::read_csv(params_csv_fallback, show_col_types = FALSE)
    if ('item' %in% names(pc)) { items <- pc$item; ok <- TRUE }
  }
  if (!ok) stop('Cannot determine item order. Provide a valid model or item parameter CSV.')
  items
}

#' Internal: validate mirt model class
#' @keywords internal
check_model <- function(mod){
  if (!methods::is(mod, 'SingleGroupClass')) stop('mod15.rds is not a mirt SingleGroupClass.')
  invisible(TRUE)
}
"
writeLines(helpers_code, file.path(pkg, "R", "helpers.R"))

# R/score_shortform.R with roxygen2
scorer_code <- "
#' Score DASS-15 and optionally approximate DASS-21
#'
#' Compute IRT EAP theta and SE from the 15-item short form, calibrate to the SEM latent,
#' apply the published ROC cut for an elevated flag, and optionally return an approximate
#' DASS-21 total and category via pre-trained links (equipercentile or linear).
#'
#' @param df Data frame with the 15 DASS short-form item columns coded 0..3, names matching the model.
#' @param cut Character. Operating point name; one of 'Youden' or 'Sens>=0.80'. Default 'Youden'.
#' @param legacy Logical. If TRUE, return approximate DASS-21 total/category. Default TRUE.
#' @param legacy_method 'equiperc' (default) or 'linear' for legacy mapping.
#' @return A data.frame with columns: theta, se, theta_cal, elevated, and optional legacy fields.
#' @examples
#' ex <- readr::read_csv(system.file('examples','example_input_20rows.csv', package='dasssf15'), show_col_types = FALSE)
#' out <- score_shortform(ex, cut='Youden', legacy=TRUE, legacy_method='equiperc')
#' head(out)
#' @export
score_shortform <- function(df, cut = c('Youden','Sens>=0.80'),
                            legacy = TRUE, legacy_method = c('equiperc','linear')){
  cut <- match.arg(cut); legacy_method <- match.arg(legacy_method)
  if (!'package:mirt' %in% search()) suppressPackageStartupMessages(library(mirt))

  # Locate assets inside package
  ext <- system.file('extdata', package = 'dasssf15')
  mod  <- readRDS(file.path(ext, 'mod15.rds')); check_model(mod)
  link <- readRDS(file.path(ext, 'link_theta_to_sem.rds'))
  ops  <- readr::read_csv(file.path(ext, 'roc_operating_points.csv'), show_col_types = FALSE)

  # Item order and input coercion
  items_order <- get_item_order(mod, params_csv_fallback = file.path(ext, 'item_parameters_min.csv'))
  to_num <- function(x){
    if (is.factor(x)) as.numeric(as.character(x))
    else if (inherits(x, 'haven_labelled')) as.numeric(as.character(haven::as_factor(x)))
    else as.numeric(x)
  }
  if (!all(items_order %in% names(df))) {
    missing <- setdiff(items_order, names(df))
    stop(sprintf('Input missing %d item(s): %s', length(missing), paste(missing, collapse = ', ')))
  }
  X <- as.data.frame(lapply(df[, items_order, drop = FALSE], to_num))
  all_na <- apply(X, 1, function(z) all(is.na(z)))
  if (any(all_na)) warning(sum(all_na), ' rows have all items missing; theta will be NA there.')

  # EAP scoring
  sc <- mirt::fscores(mod, method = 'EAP', full.scores.SE = TRUE, response.pattern = X)
  theta <- as.numeric(sc[, 'F1']); se <- as.numeric(sc[, 'SE_F1'])

  # Calibration
  theta_cal <- as.numeric(stats::predict(link, newdata = data.frame(theta = theta)))

  # ROC threshold and elevated flag
  thr <- ops$threshold[match(cut, ops$cut_name)]; if (length(thr) != 1 || is.na(thr)) thr <- ops$threshold[21]
  elevated <- ifelse(theta >= thr, 1L, 0L)

  out <- data.frame(theta = theta, se = se, theta_cal = theta_cal, elevated = elevated)

  # Approximate DASS-21 display
  if (legacy) {
    dasst <- rep(NA_real_, length(theta))
    if (legacy_method == 'equiperc' && file.exists(file.path(ext, 'legacy_link_equiperc.csv'))) {
      eq <- readr::read_csv(file.path(ext, 'legacy_link_equiperc.csv'), show_col_types = FALSE)
      p <- approx(eq$q_theta, eq$p, xout = theta, rule = 2)$y
      dasst <- approx(eq$p, eq$q_dass21, xout = p, rule = 2)$y
    } else if (legacy_method == 'linear' && file.exists(file.path(ext, 'legacy_link_linear_coeffs.csv'))) {
      co <- readr::read_csv(file.path(ext, 'legacy_link_linear_coeffs.csv'), show_col_types = FALSE)
      b0 <- co$estimate[co$term == '(Intercept)']; b1 <- co$estimate[co$term == 'theta']
      dasst <- b0 + b1 * theta
    }
    dasst <- pmax(0, round(dasst))
    cuts <- c(0,10,14,21,28)
    labs <- c('Normal','Mild','Moderate','Severe','Extremely Severe')
    cat_idx <- cut(dasst, breaks = c(-Inf, cuts[-1]-1, Inf), labels = labs, right = TRUE, include.lowest = TRUE)
    out$dass21_total_approx <- as.integer(dasst)
    out$dass21_category_approx <- as.character(cat_idx)
  }

  out
}
"
writeLines(scorer_code, file.path(pkg, "R", "score_shortform.R"))


In [25]:
# In a clean R session setwd to the package folder
#install.packages(c("devtools","roxygen2","testthat"))
devtools::document("dasssf15")   # generate NAMESPACE + man/ from roxygen2 comments
devtools::build("dasssf15")      # build source tar.gz
devtools::install("dasssf15")    # install locally
library(dasssf15)

# Smoke test on bundled example
ex <- readr::read_csv(system.file("examples","example_input_20rows.csv", package="dasssf15"), show_col_types = FALSE)
scores <- score_shortform(ex, cut = "Youden", legacy = TRUE, legacy_method = "equiperc")
head(scores); table(scores$elevated)


ℹ Updating dasssf15 documentation
ℹ Loading dasssf15
✖ Skipping ]8;;file://D:\Research files\Thesis\Smoke_mental_health\Version finals\ANALYSIS\Thesis_PositronR\dasssf15/NAMESPACENAMESPACE]8;;
ℹ It already exists and was not generated by roxygen2.


Warning message:
── Conflicts ────────────────────────────────────────────────────────────────────── dasssf15 conflicts
──
✖ `score_shortform` masks `dasssf15::score_shortform()`.
ℹ Did you accidentally source a file rather than using `load_all()`?
  Run `rm(list = c("score_shortform"))` to remove the conflicts. 


── R CMD build ──────────────────────────────────────────────────────────────────────────────────────────

Please download and install Rtools 4.5 from https://cran.r-project.org/bin/windows/Rtools/.
✔  checking for file 'D:\Research files\Thesis\Smoke_mental_health\Version finals\ANALYSIS\Thesis_PositronR\dasssf15/DESCRIPTION' (438ms)
─  preparing 'dasssf15':
✔  checking DESCRIPTION meta-information
─  checking for LF line-endings in source and make files and shell scripts
─  checking for empty or unneeded directories
   Omitted 'LazyData' from DESCRIPTION
─  building 'dasssf15_1.0.0.tar.gz'
   
These packages have more recent versions available.
It is recommended to update all of them.
Which would you like to update?

 1: All                                       
 2: CRAN packages only                        
 3: None                                      
 4: pillar       (1.10.2   -> 1.11.0  ) [CRAN]
 5: magrittr     (2.0.3    -> 2.0.4   ) [CRAN]
 6: Rcpp         (1.0.14   -> 1.1.0 

In [26]:
# 1) Clean environment and make NAMESPACE roxygen-managed
rm(list = ls())
unlink("dasssf15/NAMESPACE")   # delete old NAMESPACE
devtools::document("dasssf15") # rebuild man/ and NAMESPACE

# 2) Build and install
devtools::build("dasssf15")
devtools::install("dasssf15")
library(dasssf15)

# 3) Smoke test
ex <- readr::read_csv(system.file("examples","example_input_20rows.csv", package="dasssf15"), show_col_types = FALSE)
out <- score_shortform(ex, cut = "Youden", legacy = TRUE, legacy_method = "equiperc")
print(head(out))
print(table(out$elevated))


ℹ Updating dasssf15 documentation
ℹ Loading dasssf15
Writing ]8;;file://d:/Research files/Thesis/Smoke_mental_health/Version finals/ANALYSIS/Thesis_PositronR/NAMESPACENAMESPACE]8;;
── R CMD build ──────────────────────────────────────────────────────────────────────────────────────────

Please download and install Rtools 4.5 from https://cran.r-project.org/bin/windows/Rtools/.
✔  checking for file 'D:\Research files\Thesis\Smoke_mental_health\Version finals\ANALYSIS\Thesis_PositronR\dasssf15/DESCRIPTION' (516ms)
─  preparing 'dasssf15':
✔  checking DESCRIPTION meta-information
─  checking for LF line-endings in source and make files and shell scripts (346ms)
─  checking for empty or unneeded directories
   Omitted 'LazyData' from DESCRIPTION
─  building 'dasssf15_1.0.0.tar.gz'
   
These packages have more recent versions available.
It is recommended to update all of them.
Which would you like to update?

 1: All                                       
 2: CRAN packages only         

In [44]:
library(readr); library(dplyr); library(ggplot2)
library(dasssf15)

# 0) Load Phase 1 processed data
wide_path <- "D:/Research files/Thesis/Smoke_mental_health/Version finals/ANALYSIS/Thesis_PositronR/data/processed/analysis_phase1_ordered.rds"
dat <- readRDS(wide_path)
if (is.list(dat) && !is.data.frame(dat)) {
  if (!is.null(dat$resp_dass_num)) dat <- as.data.frame(dat$resp_dass_num) else
    if (!is.null(dat$data)) dat <- as.data.frame(dat$data) else
      stop("Could not extract a data.frame from analysis_phase1_ordered.rds")  # [2]
}

# 1) Observed DASS-21 totals from all 21 items
canon21 <- c("dQ1S","dQ2A","dQ3D","dQ4A","dQ5D","dQ6S","dQ7A","dQ8S","dQ9A","dQ10D",
             "dQ11S","dQ12S","dQ13D","dQ14S","dQ15A","dQ16D","dQ17D","dQ18S","dQ19A","dQ20A","dQ21D")
stopifnot(all(canon21 %in% names(dat)))  # ensures these exist in Phase 1 file [2]

to_num <- function(x) if (is.factor(x)) as.numeric(as.character(x)) else as.numeric(x)  # [2]
dat21 <- dat[, canon21, drop = FALSE] %>% mutate(across(everything(), to_num))  # [2]
row_na21 <- apply(dat21, 1, function(z) any(is.na(z)))  # flag rows with any NA among 21 [2]
df21 <- dat21[!row_na21, , drop = FALSE]
obs_total <- rowSums(df21, na.rm = FALSE) * 2  # DASS-21 rule [3]
cuts <- c(0,10,14,21,28); labs <- c("Normal","Mild","Moderate","Severe","Extremely Severe")  # [3]
obs_cat <- cut(obs_total, breaks = c(-Inf, cuts[-1]-1, Inf), labels = labs, right = TRUE, include.lowest = TRUE)  # [3]

# 2) Build the short-form input using the 15 required columns from the installed assets
sf_items <- readr::read_csv(system.file("extdata","item_parameters_min.csv", package="dasssf15"), show_col_types = FALSE)$item  # 15 names [1]
# Ensure all 15 exist in Phase 1 data
missing_sf <- setdiff(sf_items, names(dat))
if (length(missing_sf)) stop(sprintf("Phase 1 data is missing short-form items: %s", paste(missing_sf, collapse = ", ")))  # [2]

# Subset the same rows (non-missing on the 21 items) and only the 15 short-form columns
work <- dat[!row_na21, , drop = FALSE]
sf_df <- work[, sf_items, drop = FALSE] %>% mutate(across(everything(), to_num))  # [1]

# 3) Score the short form and approximate DASS-21 with equipercentile mapping
res <- dasssf15::score_shortform(sf_df, cut = "Youden", legacy = TRUE, legacy_method = "equiperc")  # [1][3]
pred_total <- res$dass21_total_approx
pred_cat   <- res$dass21_category_approx  # [3]

# 4) Compare observed vs approximate totals and categories
cmp <- tibble::tibble(
  obs_total = obs_total,
  obs_cat   = obs_cat,
  pred_total = pred_total,
  pred_cat   = factor(pred_cat, levels = labs)
) %>% filter(is.finite(obs_total), is.finite(pred_total))  # [3]

mae  <- mean(abs(cmp$pred_total - cmp$obs_total))  # [3]
rmse <- sqrt(mean((cmp$pred_total - cmp$obs_total)^2))  # [3]
r2   <- cor(cmp$pred_total, cmp$obs_total)^2  # [3]
metrics <- tibble::tibble(method = "equiperc", R2 = r2, MAE = mae, RMSE = rmse)  # [3]
dir.create("outputs/PhaseC_field", recursive = TRUE, showWarnings = FALSE)  # [2]
readr::write_csv(metrics, "outputs/PhaseC_field/validation_metrics_totals_phase1_direct_fixed.csv")  # [3]
print(metrics)  # [3]

# Calibration plot
p_cal <- ggplot(cmp, aes(x = pred_total, y = obs_total)) +
  geom_point(alpha = 0.25, size = 1) +
  geom_abline(slope = 1, intercept = 0, col = "grey50", lty = 2) +
  geom_smooth(method = "loess", se = TRUE, color = "#0ea5e9") +
  labs(title = "Approximate DASS-21: observed vs predicted (equiperc, fixed SF input)",
       x = "Predicted DASS-21 total", y = "Observed DASS-21 total") +
  theme_minimal(11)  # [3]
ggsave("outputs/PhaseC_field/validation_calibration_equiperc_phase1_direct_fixed.png", p_cal, width = 6.4, height = 4.5, dpi = 200)  # [3]

# Confusion matrix
tab <- table(cmp$obs_cat, cmp$pred_cat, useNA = "no")  # [3]
print(tab)
readr::write_csv(as.data.frame(tab), "outputs/PhaseC_field/validation_confusion_equiperc_phase1_direct_fixed.csv")  # [3]

# Error by decile
dec <- cut(cmp$obs_total, breaks = quantile(cmp$obs_total, probs = seq(0,1,0.1), na.rm = TRUE), include.lowest = TRUE)  # [3]
dec_err <- cmp %>%
  mutate(decile = dec, err = obs_total - pred_total) %>%
  group_by(decile) %>%
  summarise(n = n(), MAE = mean(abs(err)), Bias = mean(err), .groups = "drop")  # [3]
print(dec_err)
readr::write_csv(dec_err, "outputs/PhaseC_field/validation_error_by_decile_phase1_direct_fixed.csv")  # [3]


# A tibble: 1 × 4
  method      R2   MAE  RMSE
  <chr>    <dbl> <dbl> <dbl>
1 equiperc 0.896  6.38  8.10
`geom_smooth()` using formula = 'y ~ x'
                  
                   Normal Mild Moderate Severe Extremely Severe
  Normal               14    2        3      0                0
  Mild                  2    3        3      0                0
  Moderate              5    2       20      6                2
  Severe                1    1        9     15               17
  Extremely Severe      0    1       12     20              531
# A tibble: 10 × 4
   decile        n   MAE   Bias
   <fct>     <int> <dbl>  <dbl>
 1 [4,22]       79  4.03 -0.785
 2 (22,30]      57  6.30 -1.53 
 3 (30,38]      69  7.17 -0.681
 4 (38,44.4]    63  7     1.63 
 5 (44.4,52]    73  7.40  2.08 
 6 (52,58]      62  6.08  0.597
 7 (58,66]      83  7.39  3.19 
 8 (66,74]      53  5.42  2.32 
 9 (74,86]      64  6.89  2.45 
10 (86,120]     66  6.05  3.62 


In [56]:
library(readr); library(dplyr); library(ggplot2)
library(dasssf15)

# Load your Phase 1 processed data
wide_path <- "D:/Research files/Thesis/Smoke_mental_health/Version finals/ANALYSIS/Thesis_PositronR/data/processed/analysis_phase1_ordered.rds"
dat <- readRDS(wide_path)
to_num <- function(x) if (is.factor(x)) as.numeric(as.character(x)) else as.numeric(x)

# DASS-21 canonical columns
canon21 <- c("dQ1S","dQ2A","dQ3D","dQ4A","dQ5D","dQ6S","dQ7A","dQ8S","dQ9A","dQ10D",
             "dQ11S","dQ12S","dQ13D","dQ14S","dQ15A","dQ16D","dQ17D","dQ18S","dQ19A","dQ20A","dQ21D")
dat21 <- dat[, canon21, drop = FALSE] %>% mutate(across(everything(), to_num))
row_na21 <- apply(dat21, 1, function(z) any(is.na(z)))
df21 <- dat21[!row_na21, , drop = FALSE]
obs_total <- rowSums(df21, na.rm = FALSE) * 2
cuts <- c(0,10,14,21,28)
labs <- c("Normal","Mild","Moderate","Severe","Extremely Severe")
obs_cat <- cut(obs_total, breaks = c(-Inf, cuts[-1]-1, Inf), labels = labs, right = TRUE, include.lowest = TRUE)

# **HARD CODED DASS-15 item order below**
sf_items <- c(
  "dQ7A",
  "dQ10D",
  "dQ13D",
  "dQ14S",
  "dQ16D",
  "dQ17D",
  "dQ18S",
  "dQ19A",
  "dQ20A",
  "dQ21D",
  "dQ1S",
  "dQ9A",
  "dQ11S",
  "dQ12S",
  "dQ15A"
)
stopifnot(all(sf_items %in% names(dat))) # ensures all columns exist

work <- dat[!row_na21, , drop = FALSE]
sf_df <- work[, sf_items, drop = FALSE] %>% mutate(across(everything(), to_num))

res <- dasssf15::score_shortform(sf_df, cut = "Youden", legacy = TRUE, legacy_method = "equiperc")
pred_total <- res$dass21_total_approx
pred_cat   <- res$dass21_category_approx

cmp <- tibble::tibble(
  obs_total = obs_total,
  obs_cat   = obs_cat,
  pred_total = pred_total,
  pred_cat   = factor(pred_cat, levels = labs)
) %>% filter(is.finite(obs_total), is.finite(pred_total))

# Now compute "Severe" agreement as previously shown
obs_sev <- cmp$obs_cat == "Severe"
pred_sev <- cmp$pred_cat == "Severe"

tp <- sum(obs_sev & pred_sev, na.rm = TRUE)
tn <- sum(!obs_sev & !pred_sev, na.rm = TRUE)
fp <- sum(!obs_sev & pred_sev, na.rm = TRUE)
fn <- sum(obs_sev & !pred_sev, na.rm = TRUE)

sensitivity <- tp / (tp + fn)
specificity <- tn / (tn + fp)
ppv <- tp / (tp + fp)
npv <- tn / (tn + fn)
accuracy <- (tp + tn) / (tp + tn + fp + fn)
f1 <- ifelse((ppv + sensitivity) > 0, 2 * ppv * sensitivity / (ppv + sensitivity), NA)

severe_metrics <- data.frame(tp, fp, fn, tn, sensitivity, specificity, ppv, npv, accuracy, f1)
print(severe_metrics)

tab5 <- table(cmp$obs_cat, cmp$pred_cat, useNA = "no")
print(tab5["Severe", ])
print(tab5[, "Severe"])


  tp fp fn  tn sensitivity specificity       ppv      npv  accuracy        f1
1 15 26 28 600   0.3488372   0.9584665 0.3658537 0.955414 0.9192825 0.3571429
          Normal             Mild         Moderate           Severe Extremely Severe 
               1                1                9               15               17 
          Normal             Mild         Moderate           Severe Extremely Severe 
               0                0                6               15               20 


In [57]:
library(readr); library(dplyr)

# Load your Phase 1 data
wide_path <- "D:/Research files/Thesis/Smoke_mental_health/Version finals/ANALYSIS/Thesis_PositronR/data/processed/analysis_phase1_ordered.rds"
dat <- readRDS(wide_path)
to_num <- function(x) if (is.factor(x)) as.numeric(as.character(x)) else as.numeric(x)

# Canonical column lists
canon21 <- c("dQ1S","dQ2A","dQ3D","dQ4A","dQ5D","dQ6S","dQ7A","dQ8S","dQ9A","dQ10D",
             "dQ11S","dQ12S","dQ13D","dQ14S","dQ15A","dQ16D","dQ17D","dQ18S","dQ19A","dQ20A","dQ21D")
sf_items <- c("dQ7A","dQ10D","dQ13D","dQ14S","dQ16D","dQ17D","dQ18S","dQ19A","dQ20A",
              "dQ21D","dQ1S","dQ9A","dQ11S","dQ12S","dQ15A")
stopifnot(all(canon21 %in% names(dat)), all(sf_items %in% names(dat)))

# Compute true DASS-21 total/band and DASS-15 raw sum
dat21 <- dat[, canon21, drop = FALSE] %>% mutate(across(everything(), to_num))
sfmat <- dat[, sf_items, drop = FALSE] %>% mutate(across(everything(), to_num))
row_na21 <- apply(dat21, 1, function(z) any(is.na(z)))
row_nasf <- apply(sfmat, 1, function(z) any(is.na(z)))
keep <- !row_na21 & !row_nasf

obs_total <- rowSums(dat21[keep, , drop=FALSE], na.rm = FALSE) * 2
sf_raw <- rowSums(sfmat[keep, , drop=FALSE], na.rm = FALSE)

# DASS-21 bands (for use as Gold Standard)
cuts <- c(0,10,14,21,28)
labs <- c("Normal","Mild","Moderate","Severe","Extremely Severe")
obs_cat <- cut(obs_total, breaks = c(-Inf, cuts[-1]-1, Inf), labels = labs, right = TRUE, include.lowest = TRUE)

# ---- (A) Scientific process: DASS-15 IRT → equipercentile → DASS-21 ----
library(dasssf15)
res <- dasssf15::score_shortform(sfmat[keep,], cut = "Youden", legacy = TRUE, legacy_method = "equiperc")
irt_total <- res$dass21_total_approx
irt_cat   <- factor(res$dass21_category_approx, levels = labs)

# ---- (B) Legacy ratio process: DASS-15 raw sum * ratio → DASS-42 ----
# Legacy DASS-21: multiply by 2.8 (to 42 max) or by 2 (like DASS-21)
dass15_to_21 <- sf_raw * (21/15)
dass15_to_42 <- sf_raw * (42/15)

# Assign bands for "legacy DASS-21"
legacy21_cat <- cut(dass15_to_21, breaks = c(-Inf, cuts[-1]-1, Inf), labels = labs, right = TRUE, include.lowest = TRUE)

# For DASS-42 (using common DASS-42 reference bands)
cuts42 <- c(0,14,18,26,34)
labs42 <- c("Normal","Mild","Moderate","Severe","Extremely Severe")
legacy42_cat <- cut(dass15_to_42, breaks = c(-Inf, cuts42[-1]-1, Inf), labels = labs42, right = TRUE, include.lowest = TRUE)

# ---- Accuracy Reports against true DASS-21 ----
tab_irt <- table(obs_cat, irt_cat, useNA="no")
tab_legacy <- table(obs_cat, legacy21_cat, useNA="no")
print(tab_irt)
print(tab_legacy)

# Optionally, for DASS-42
# obs42_total <- obs_total * 2   # assumes DASS-21 total ×2 as DASS-42 proxy
# obs42_cat <- cut(obs42_total, breaks = c(-Inf, cuts42[-1]-1, Inf), labels = labs42, right = TRUE, include.lowest = TRUE)
# tab_legacy42 <- table(obs42_cat, legacy42_cat, useNA="no")
# print(tab_legacy42)

# Overall and Severe-band agreement
get_match_prop <- function(obs, pred, level) mean(obs == pred & obs == level, na.rm = TRUE)
severe_match_irt    <- get_match_prop(obs_cat, irt_cat, "Severe")
severe_match_legacy <- get_match_prop(obs_cat, legacy21_cat, "Severe")
cat(sprintf("Fraction of true 'Severe' matched by IRT: %.2f\n", severe_match_irt))
cat(sprintf("Fraction of true 'Severe' matched by legacy ratio: %.2f\n", severe_match_legacy))

# Save outputs
dir.create("outputs/PhaseC_field", recursive = TRUE, showWarnings = FALSE)
write_csv(data.frame(irt=irt_cat, legacy=legacy21_cat, truth=obs_cat), "outputs/PhaseC_field/legacy_vs_linked_bands.csv")
write_csv(as.data.frame(tab_irt), "outputs/PhaseC_field/confusion_irt_vs_dass21.csv")
write_csv(as.data.frame(tab_legacy), "outputs/PhaseC_field/confusion_legacy_vs_dass21.csv")


                  irt_cat
obs_cat            Normal Mild Moderate Severe Extremely Severe
  Normal               14    2        3      0                0
  Mild                  2    3        3      0                0
  Moderate              5    2       20      6                2
  Severe                1    1        9     15               17
  Extremely Severe      0    1       12     20              531
                  legacy21_cat
obs_cat            Normal Mild Moderate Severe Extremely Severe
  Normal               19    0        0      0                0
  Mild                  8    0        0      0                0
  Moderate             26    9        0      0                0
  Severe                7   22       14      0                0
  Extremely Severe      2   18      123    124              297
Fraction of true 'Severe' matched by IRT: 0.02
Fraction of true 'Severe' matched by legacy ratio: 0.00


In [ ]:
#FINAL Practical Use Files

In [45]:
# From the project root
# 1) Clean session and ensure devtools/roxygen2 installed
rm(list = ls())  # clear objects that might mask package functions
if (!requireNamespace("devtools", quietly = TRUE)) install.packages("devtools")
if (!requireNamespace("roxygen2", quietly = TRUE)) install.packages("roxygen2")

# 2) Remove manual NAMESPACE and regenerate via roxygen2
ns_path <- "dasssf15/NAMESPACE"
if (file.exists(ns_path)) unlink(ns_path)
devtools::document("dasssf15")   # rebuild NAMESPACE + man/ from roxygen headers [1]

# 3) Build and install locally
devtools::build("dasssf15")      # creates source tar.gz [2]
devtools::install("dasssf15")    # installs package [2]

# 4) Load and confirm help exists
library(dasssf15)
help("score_shortform")  # should open the man page generated by roxygen2 [1]


ℹ Updating dasssf15 documentation
ℹ Loading dasssf15
Writing ]8;;file://d:/Research files/Thesis/Smoke_mental_health/Version finals/ANALYSIS/Thesis_PositronR/NAMESPACENAMESPACE]8;;
── R CMD build ───────────────────────────────────────────────────────────────────────────────────────────────────

Please download and install Rtools 4.5 from https://cran.r-project.org/bin/windows/Rtools/.
✔  checking for file 'D:\Research files\Thesis\Smoke_mental_health\Version finals\ANALYSIS\Thesis_PositronR\dasssf15/DESCRIPTION' (1.6s)
─  preparing 'dasssf15':
✔  checking DESCRIPTION meta-information
─  checking for LF line-endings in source and make files and shell scripts (429ms)
─  checking for empty or unneeded directories
   Omitted 'LazyData' from DESCRIPTION
─  building 'dasssf15_1.0.0.tar.gz'
   
These packages have more recent versions available.
It is recommended to update all of them.
Which would you like to update?

 1: All                                       
 2: CRAN packages only 

In [46]:
# After installing Rtools 4.5, verify build tools are available
if (!requireNamespace("pkgbuild", quietly = TRUE)) install.packages("pkgbuild")
pkgbuild::has_build_tools(debug = TRUE)  # should return TRUE with path info [2]


Trying to compile a simple C file
Running "C:/Program Files/R/R-4.5.1/bin/x64/Rcmd.exe" SHLIB foo.c
Error in system(paste(MAKE, p1(paste("-f", shQuote(makefiles))), "compilers"),  : 
  'make' not found
Calls: <Anonymous> -> .shlib_internal -> system
Execution halted

Please download and install the appropriate version of Rtools for 4.5.1 from
https://cran.r-project.org/bin/windows/Rtools/.


In [47]:
# File: dasssf15/R/validate.R

#' Validate Approximate DASS-21 on a dataset with full DASS-21 items
#'
#' Given a data frame that contains the 21 canonical DASS-21 item columns
#' (dQ1S..dQ21D) and the 15 short-form columns, compute observed totals/bands,
#' run short-form scoring with equipercentile legacy mapping, and report accuracy.
#'
#' @param df A data.frame with columns dQ1S..dQ21D and the 15 short-form items.
#' @param out_dir Directory to write calibration plot and CSV outputs.
#' @return A list with metrics (R2, MAE, RMSE), confusion table (as a matrix),
#'         and paths to written files (metrics_csv, confusion_csv, decile_csv, cal_plot).
#' @export
validate_on_sample <- function(df, out_dir = "outputs/PhaseC_field"){
  if (!"package:mirt" %in% search()) suppressPackageStartupMessages(library(mirt))  # fscores dependency [3]

  # 21 and 15 canonical sets
  canon21 <- c("dQ1S","dQ2A","dQ3D","dQ4A","dQ5D","dQ6S","dQ7A","dQ8S","dQ9A","dQ10D",
               "dQ11S","dQ12S","dQ13D","dQ14S","dQ15A","dQ16D","dQ17D","dQ18S","dQ19A","dQ20A","dQ21D")
  sf_items <- c("dQ1S","dQ2A","dQ3D","dQ4A","dQ5D","dQ6S","dQ7A","dQ8S","dQ9A","dQ10D",
                "dQ11S","dQ12S","dQ13D","dQ14S","dQ15A")

  miss21 <- setdiff(canon21, names(df))
  miss15 <- setdiff(sf_items, names(df))
  if (length(miss21)) stop("Missing DASS-21 item columns: ", paste(miss21, collapse = ", "))  # [5]
  if (length(miss15)) stop("Missing short-form item columns: ", paste(miss15, collapse = ", "))  # [5]

  to_num <- function(x){
    if (is.factor(x)) as.numeric(as.character(x))
    else if (inherits(x,"haven_labelled")) as.numeric(as.character(haven::as_factor(x)))
    else as.numeric(x)
  }

  # Observed totals and categories from 21 items
  dat21 <- df[, canon21, drop = FALSE]
  dat21 <- as.data.frame(lapply(dat21, to_num))
  row_na21 <- apply(dat21, 1, function(z) any(is.na(z)))
  df21 <- dat21[!row_na21, , drop = FALSE]
  obs_total <- rowSums(df21, na.rm = FALSE) * 2  # DASS-21 rule [4]

  cuts <- c(0,10,14,21,28); labs <- c("Normal","Mild","Moderate","Severe","Extremely Severe")  # [4]
  obs_cat <- cut(obs_total, breaks = c(-Inf, cuts[-1]-1, Inf), labels = labs, right = TRUE, include.lowest = TRUE)  # [4]

  # Short-form scoring and equipercentile legacy mapping
  sf_df <- df[!row_na21, sf_items, drop = FALSE]
  sf_df <- as.data.frame(lapply(sf_df, to_num))
  res <- score_shortform(sf_df, cut = "Youden", legacy = TRUE, legacy_method = "equiperc")  # [3][4]
  pred_total <- res$dass21_total_approx; pred_cat <- factor(res$dass21_category_approx, levels = labs)  # [4]

  # Metrics, confusion, decile errors
  keep <- is.finite(obs_total) & is.finite(pred_total)
  obs <- obs_total[keep]; prd <- pred_total[keep]
  mae  <- mean(abs(prd - obs))  # [4]
  rmse <- sqrt(mean((prd - obs)^2))  # [4]
  r2   <- cor(prd, obs)^2  # [4]
  metrics <- data.frame(method = "equiperc", R2 = r2, MAE = mae, RMSE = rmse)  # [4]

  tab <- table(obs_cat[keep], pred_cat[keep], useNA = "no")  # [4]

  dec <- cut(obs, breaks = quantile(obs, probs = seq(0,1,0.1), na.rm = TRUE), include.lowest = TRUE)  # [4]
  err <- obs - prd
  dec_err <- aggregate(cbind(MAE = abs(err), Bias = err), by = list(decile = dec), FUN = mean)
  dec_err$n <- as.integer(table(dec))
  dec_err <- dec_err[, c("decile","n","MAE","Bias")]

  # Write outputs
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)  # [2]
  metrics_csv  <- file.path(out_dir, "validation_metrics_totals_package.csv")
  confusion_csv<- file.path(out_dir, "validation_confusion_equiperc_package.csv")
  decile_csv   <- file.path(out_dir, "validation_error_by_decile_package.csv")
  cal_plot     <- file.path(out_dir, "validation_calibration_equiperc_package.png")

  readr::write_csv(metrics, metrics_csv)  # [4]
  readr::write_csv(as.data.frame(tab), confusion_csv)  # [4]
  readr::write_csv(dec_err, decile_csv)  # [4]

  # Calibration plot
  suppressPackageStartupMessages(library(ggplot2))
  p <- ggplot(data.frame(obs = obs, pred = prd), aes(x = pred, y = obs)) +
    geom_point(alpha = 0.25, size = 1) +
    geom_abline(slope = 1, intercept = 0, col = "grey50", lty = 2) +
    geom_smooth(method = "loess", se = TRUE, color = "#0ea5e9") +
    labs(title = "Approximate DASS-21: observed vs predicted (equiperc)",
         x = "Predicted DASS-21 total", y = "Observed DASS-21 total") +
    theme_minimal(11)  # [4]
  ggsave(cal_plot, p, width = 6.4, height = 4.5, dpi = 200)  # [4]

  list(metrics = metrics, confusion = tab,
       files = list(metrics_csv = metrics_csv, confusion_csv = confusion_csv,
                    decile_csv = decile_csv, cal_plot = cal_plot))
}


In [48]:
# Smoke test after a clean install (on another machine)
rm(list = ls())
library(dasssf15)

# 1) Example scoring
ex <- readr::read_csv(system.file("examples","example_input_20rows.csv", package="dasssf15"), show_col_types = FALSE)
out <- score_shortform(ex, cut = "Youden", legacy = TRUE, legacy_method = "equiperc")  # [3][4]
stopifnot(all(c("theta","se","theta_cal","elevated") %in% names(out)))  # [3]
print(head(out))

# 2) Validator (if a dataset with 21 items is accessible)
# df21 <- readRDS("path/to/analysis_phase1_ordered.rds")  # ensure it includes dQ1S..dQ21D
# val <- validate_on_sample(df21)
# print(val$metrics)


       theta        se   theta_cal elevated dass21_total_approx dass21_category_approx
1  0.3882666 0.3423946  0.39866151        1                  62       Extremely Severe
2 -0.2930697 0.4375308 -0.25768590        0                  42       Extremely Severe
3  0.1815308 0.4147761  0.19950801        1                  58       Extremely Severe
4 -1.5736995 0.4364924 -1.49134690        0                  14               Moderate
5 -0.1076427 0.3469108 -0.07905966        0                  48       Extremely Severe
6 -0.1219550 0.3318533 -0.09284702        0                  48       Extremely Severe
